# 00 – Verifica allineamento EC3D con il paper

**Obiettivo**: Verificare che i dati EC3D nel repo corrispondano alla tabella del paper.

Output attesi:
1. Tabella per-SUBJECT e per-INSTRUCTION identica al paper
2. Tabella aggregata per-ACTION (totali 132/127/103) e totale complessivo (362)
3. Conteggio e dettaglio degli "unknown" (esempi non presenti nel paper)
4. Maschera/filtro `no_unknown` per esperimenti puliti


In [ ]:
import pickle
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Cartella root del progetto
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data" / "EC3D"

print("ROOT_DIR:", ROOT_DIR)
print("DATA_DIR:", DATA_DIR)


## A) Caricamento split e mapping Subject

Nel paper:
- Subject 1, 2, 3 → Training
- Subject 4 → Test

Dal file `split_cross_subject.json`:
- train_subjects: ["Hugues", "Sena", "Isinsu"]
- test_subjects: ["Vidit"]


In [ ]:
# Carica lo split
SPLIT_PATH = DATA_DIR / "split_cross_subject.json"
with open(SPLIT_PATH, "r") as f:
    split_data = json.load(f)

print("Train subjects:", split_data["train_subjects"])
print("Test subjects:", split_data["test_subjects"])

# Mapping Subject Name -> Subject Index (1-based come nel paper)
# ATTENZIONE: L'ordine CORRETTO verificato confrontando i conteggi col paper è:
#   Subject 1 = Hugues
#   Subject 2 = Isinsu  (NON Sena!)
#   Subject 3 = Sena    (NON Isinsu!)
#   Subject 4 = Vidit
SUBJECT_NAME_TO_IDX = {
    "Hugues": 1,
    "Isinsu": 2,  # Corretto: Isinsu è Subject 2
    "Sena": 3,    # Corretto: Sena è Subject 3
    "Vidit": 4,
}

SUBJECT_IDX_TO_NAME = {v: k for k, v in SUBJECT_NAME_TO_IDX.items()}

print("\nMapping Subject (verificato con paper):")
for name, idx in sorted(SUBJECT_NAME_TO_IDX.items(), key=lambda x: x[1]):
    role = "train" if name in split_data["train_subjects"] else "test"
    print(f"  Subject {idx} = {name} ({role})")


## B) Caricamento dati EC3D e estrazione metadati sequenze

Carichiamo `data_3D.pickle` per avere accesso a tutti i frame con le loro label.


In [ ]:
# Carica data_3D.pickle
DATA_3D_PATH = DATA_DIR / "data_3D.pickle"
with open(DATA_3D_PATH, "rb") as f:
    data_3d = pickle.load(f)

poses = data_3d["poses"]   # (N_frames, 3, 25)
labels = data_3d["labels"] # (N_frames, 5) - [exercise, subject, instruction_id, trial_id, frame_idx]

print(f"poses shape: {poses.shape}")
print(f"labels shape: {labels.shape}")
print(f"\nPrime 5 label: {labels[:5]}")


In [ ]:
# Creo DataFrame dei frame
labels_df = pd.DataFrame(
    labels,
    columns=["exercise", "subject", "instruction_id_str", "trial_id_str", "frame_idx_str"]
)

# Converto tipi
labels_df["instruction_id"] = labels_df["instruction_id_str"].astype(int)
labels_df["trial_id"] = labels_df["trial_id_str"].astype(int)
labels_df["frame_idx"] = labels_df["frame_idx_str"].astype(int)
labels_df["subject_idx"] = labels_df["subject"].map(SUBJECT_NAME_TO_IDX)

# Normalizza nomi esercizi (nel file: SQUAT, Lunges, Plank → nel paper: Squats, Lunges, Planks)
EXERCISE_NORMALIZE = {
    "SQUAT": "Squats",
    "Lunges": "Lunges",
    "Plank": "Planks",
}
labels_df["exercise_paper"] = labels_df["exercise"].map(EXERCISE_NORMALIZE)

print("Esercizi unici (raw):", labels_df["exercise"].unique())
print("Esercizi unici (paper):", labels_df["exercise_paper"].unique())
print("\nInstruction IDs unici:", sorted(labels_df["instruction_id"].unique()))
print("Subjects unici:", labels_df["subject"].unique())


## C) Definizione tassonomia del paper

Dal paper:
- **Squats**: 1=Correct, 2=Feet too wide, 3=Knees inward, 4=Not low enough, 5=Front bent
- **Lunges**: 1=Correct, 4=Not low enough, 6=Knee passes toe
- **Planks**: 1=Correct, 7=Arched back, 8=Hunch back

Nel codice originale:
- 7=Banana back → nel paper = Arched back
- 8=Rolled back → nel paper = Hunch back


In [ ]:
# Tassonomia PAPER: (exercise_paper, instruction_id) -> instruction_label_paper
PAPER_TAXONOMY = {
    # Squats
    ("Squats", 1): "Correct",
    ("Squats", 2): "Feet too wide",
    ("Squats", 3): "Knees inward",
    ("Squats", 4): "Not low enough",
    ("Squats", 5): "Front bent",
    # Lunges
    ("Lunges", 1): "Correct",
    ("Lunges", 4): "Not low enough",
    ("Lunges", 6): "Knee passes toe",
    # Planks
    ("Planks", 1): "Correct",
    ("Planks", 7): "Arched back",
    ("Planks", 8): "Hunch back",
}

# Mapping alternativo con nomi RAW usati nel codice
RAW_INSTRUCTION_NAMES = {
    1: "Correct",
    2: "Feets too wide",  # typo originale nel codice
    3: "Knees inward",
    4: "Not low enough",
    5: "Front bended",    # variante originale
    6: "Knees pass toes",
    7: "Banana back",     # nel paper = Arched back
    8: "Rolled back",     # nel paper = Hunch back
    9: "Asymmetric",      # non presente nei dati
    10: "Unknown",        # NON nel paper!
}

# Conteggi attesi dal paper per ogni (exercise, instruction)
PAPER_EXPECTED_COUNTS = {
    # Squats - totale 132
    ("Squats", "Correct"): {1: 10, 2: 10, 3: 11, 4: 10},        # tot 41
    ("Squats", "Feet too wide"): {1: 5, 2: 8, 3: 5, 4: 5},      # tot 23
    ("Squats", "Knees inward"): {1: 6, 2: 7, 3: 5, 4: 5},       # tot 23
    ("Squats", "Not low enough"): {1: 5, 2: 7, 3: 5, 4: 4},     # tot 21
    ("Squats", "Front bent"): {1: 5, 2: 6, 3: 6, 4: 7},         # tot 24
    # Lunges - totale 127
    ("Lunges", "Correct"): {1: 12, 2: 11, 3: 11, 4: 12},        # tot 46
    ("Lunges", "Not low enough"): {1: 10, 2: 10, 3: 10, 4: 10}, # tot 40
    ("Lunges", "Knee passes toe"): {1: 10, 2: 10, 3: 11, 4: 10},# tot 41
    # Planks - totale 103
    ("Planks", "Correct"): {1: 7, 2: 8, 3: 11, 4: 7},           # tot 33
    ("Planks", "Arched back"): {1: 5, 2: 5, 3: 11, 4: 9},       # tot 30
    ("Planks", "Hunch back"): {1: 10, 2: 10, 3: 11, 4: 9},      # tot 40
}

# Totali per action dal paper
PAPER_TOTALS_PER_ACTION = {
    "Squats": 132,
    "Lunges": 127,
    "Planks": 103,
}

PAPER_TOTAL_OVERALL = 362

print("Tassonomia paper definita con", len(PAPER_TAXONOMY), "combinazioni (exercise, instruction)")
print("Totali attesi per action:", PAPER_TOTALS_PER_ACTION)
print("Totale complessivo atteso:", PAPER_TOTAL_OVERALL)


## D) Creazione DataFrame sequenze con tutte le info


In [ ]:
# Aggiungo instruction_label dal paper (se presente nella tassonomia)
def get_paper_label(row):
    key = (row["exercise_paper"], row["instruction_id"])
    return PAPER_TAXONOMY.get(key, None)

def get_raw_label(row):
    return RAW_INSTRUCTION_NAMES.get(row["instruction_id"], f"ID_{row['instruction_id']}")

labels_df["instruction_label_paper"] = labels_df.apply(get_paper_label, axis=1)
labels_df["instruction_label_raw"] = labels_df.apply(get_raw_label, axis=1)

# Flag: è nella tassonomia del paper?
labels_df["is_in_paper_taxonomy"] = labels_df["instruction_label_paper"].notna()

print("\nFrame nella tassonomia paper:", labels_df["is_in_paper_taxonomy"].sum())
print("Frame NON nella tassonomia (unknown):", (~labels_df["is_in_paper_taxonomy"]).sum())


In [ ]:
# Costruisco DataFrame delle SEQUENZE (non frame)
# Chiave sequenza: (exercise, subject, instruction_id, trial_id)
# NOTA: fillna per evitare che groupby escluda le righe con instruction_label_paper=None

labels_df["instruction_label_paper_filled"] = labels_df["instruction_label_paper"].fillna("__UNKNOWN__")

seq_df = (
    labels_df
    .groupby(["exercise", "exercise_paper", "subject", "subject_idx", 
              "instruction_id", "instruction_label_paper_filled", "instruction_label_raw",
              "is_in_paper_taxonomy", "trial_id"])
    .agg(n_frames=("frame_idx", "count"))
    .reset_index()
)

# Ripristina None per le etichette unknown
seq_df["instruction_label_paper"] = seq_df["instruction_label_paper_filled"].replace("__UNKNOWN__", None)

print(f"\nTotale sequenze: {len(seq_df)}")
print(f"Sequenze in tassonomia paper: {seq_df['is_in_paper_taxonomy'].sum()}")
print(f"Sequenze UNKNOWN: {(~seq_df['is_in_paper_taxonomy']).sum()}")

seq_df.head(10)


## E) Pivot Table come nel paper


In [ ]:
# Filtra solo le sequenze nella tassonomia paper
seq_paper = seq_df[seq_df["is_in_paper_taxonomy"]].copy()

# Conteggio per (exercise_paper, instruction_label_paper, subject_idx)
pivot_data = (
    seq_paper
    .groupby(["exercise_paper", "instruction_label_paper", "subject_idx"])
    .size()
    .reset_index(name="count")
)

# Pivot: righe = (exercise, instruction), colonne = Subject 1..4
pivot_table = pivot_data.pivot_table(
    index=["exercise_paper", "instruction_label_paper"],
    columns="subject_idx",
    values="count",
    fill_value=0,
    aggfunc="sum"
)

# Rinomina colonne
pivot_table.columns = [f"Subject {i}" for i in pivot_table.columns]

# Aggiungo Total (per instruction)
pivot_table["Total (per instruction)"] = pivot_table.sum(axis=1)

pivot_table


In [ ]:
# Ordino le righe come nel paper
PAPER_ORDER = [
    ("Squats", "Correct"),
    ("Squats", "Feet too wide"),
    ("Squats", "Knees inward"),
    ("Squats", "Not low enough"),
    ("Squats", "Front bent"),
    ("Lunges", "Correct"),
    ("Lunges", "Not low enough"),
    ("Lunges", "Knee passes toe"),
    ("Planks", "Correct"),
    ("Planks", "Arched back"),
    ("Planks", "Hunch back"),
]

# Reindex per ordinare
pivot_ordered = pivot_table.reindex(PAPER_ORDER)

# Aggiungo Total (per action) - mostrando solo sulla prima riga di ogni exercise
totals_per_action = pivot_ordered.groupby(level=0)["Total (per instruction)"].sum()

print("="*80)
print("TABELLA CONFRONTO CON PAPER - Conteggi per Subject e Instruction")
print("="*80)
print(pivot_ordered.to_string())
print("\n" + "="*80)
print("TOTALI PER ACTION:")
print(totals_per_action.to_string())
print(f"\nTOTALE COMPLESSIVO: {pivot_ordered['Total (per instruction)'].sum()}")
print("="*80)


In [ ]:
# Creo tabella formattata IDENTICA al paper (con Total per action)
def create_paper_style_table(pivot_df):
    """Crea tabella stile paper con Total per action."""
    rows = []
    current_exercise = None
    exercise_rows = {}
    
    for (ex, instr) in PAPER_ORDER:
        if ex not in exercise_rows:
            exercise_rows[ex] = []
        row = pivot_df.loc[(ex, instr)].to_dict()
        row["Exercise"] = ex
        row["Instruction Label"] = instr
        exercise_rows[ex].append(row)
    
    # Calcola totali per action
    result_rows = []
    for ex in ["Squats", "Lunges", "Planks"]:
        ex_total = sum(r["Total (per instruction)"] for r in exercise_rows[ex])
        for i, row in enumerate(exercise_rows[ex]):
            row["Total (per action)"] = ex_total if i == 0 else ""
            result_rows.append(row)
    
    cols = ["Exercise", "Instruction Label", "Subject 1", "Subject 2", "Subject 3", "Subject 4", 
            "Total (per instruction)", "Total (per action)"]
    return pd.DataFrame(result_rows)[cols]

paper_table = create_paper_style_table(pivot_ordered)

print("\n" + "="*100)
print("TABELLA STILE PAPER (con Total per action)")
print("="*100)
print(paper_table.to_string(index=False))
print("="*100)


In [ ]:
# Confronto dettagliato con valori attesi dal paper
print("\n" + "="*80)
print("VERIFICA CONFRONTO CON PAPER")
print("="*80)

mismatches = []
for (ex, instr), expected_per_subj in PAPER_EXPECTED_COUNTS.items():
    for subj_idx, expected_count in expected_per_subj.items():
        actual_count = pivot_ordered.loc[(ex, instr), f"Subject {subj_idx}"]
        if actual_count != expected_count:
            mismatches.append({
                "Exercise": ex,
                "Instruction": instr,
                "Subject": subj_idx,
                "Expected (paper)": expected_count,
                "Actual (data)": actual_count,
                "Diff": actual_count - expected_count,
            })

if mismatches:
    print(f"\n⚠️  TROVATI {len(mismatches)} MISMATCH con il paper:\n")
    mismatch_df = pd.DataFrame(mismatches)
    print(mismatch_df.to_string(index=False))
else:
    print("\n✅ TUTTI I CONTEGGI CORRISPONDONO AL PAPER!")

# Verifica totali per action
print("\n" + "-"*50)
print("Verifica totali per action:")
for ex, expected_total in PAPER_TOTALS_PER_ACTION.items():
    actual_total = totals_per_action[ex]
    status = "✅" if actual_total == expected_total else "❌"
    print(f"  {ex}: atteso={expected_total}, trovato={actual_total} {status}")

overall_total = pivot_ordered["Total (per instruction)"].sum()
status = "✅" if overall_total == PAPER_TOTAL_OVERALL else "❌"
print(f"\n  TOTALE OVERALL: atteso={PAPER_TOTAL_OVERALL}, trovato={overall_total} {status}")


## F) Analisi degli UNKNOWN (fuori tassonomia paper)


In [ ]:
# Sequenze UNKNOWN (non nella tassonomia paper)
seq_unknown = seq_df[~seq_df["is_in_paper_taxonomy"]].copy()

print("="*80)
print("ANALISI SEQUENZE UNKNOWN (non nel paper)")
print("="*80)

print(f"\nTotale sequenze UNKNOWN: {len(seq_unknown)}")
print(f"Totale frame UNKNOWN: {seq_unknown['n_frames'].sum()}")


In [ ]:
# Dettaglio: quali (exercise, instruction_id) sono unknown?
unknown_combos = (
    seq_unknown
    .groupby(["exercise", "exercise_paper", "instruction_id", "instruction_label_raw"])
    .agg(
        n_sequences=("trial_id", "count"),
        n_frames=("n_frames", "sum")
    )
    .reset_index()
)

print("\n" + "-"*50)
print("Combinazioni (exercise, instruction_id) UNKNOWN:")
print(unknown_combos.to_string(index=False))


In [ ]:
# Pivot degli unknown per subject
if len(seq_unknown) > 0:
    unknown_pivot = (
        seq_unknown
        .groupby(["exercise_paper", "instruction_label_raw", "subject_idx"])
        .size()
        .reset_index(name="count")
        .pivot_table(
            index=["exercise_paper", "instruction_label_raw"],
            columns="subject_idx",
            values="count",
            fill_value=0
        )
    )
    unknown_pivot.columns = [f"Subject {i}" for i in unknown_pivot.columns]
    unknown_pivot["Total"] = unknown_pivot.sum(axis=1)
    
    print("\n" + "-"*50)
    print("UNKNOWN per Subject:")
    print(unknown_pivot.to_string())
else:
    print("\nNessuna sequenza unknown trovata.")


In [ ]:
# Esempi di sequenze unknown (prime 10)
print("\n" + "-"*50)
print("Esempi di sequenze UNKNOWN (prime 10):")
unknown_examples = seq_unknown.head(10)[["exercise_paper", "subject", "subject_idx", 
                                          "instruction_id", "instruction_label_raw",
                                          "trial_id", "n_frames"]]
print(unknown_examples.to_string(index=False))


## G) Creazione dataset df_full e df_no_unknown


In [ ]:
# df_full: tutte le sequenze
df_full = seq_df.copy()

# df_no_unknown: solo sequenze nella tassonomia paper
df_no_unknown = seq_df[seq_df["is_in_paper_taxonomy"]].copy()

print("="*80)
print("CONFRONTO DATASET COMPLETO vs NO_UNKNOWN")
print("="*80)

print(f"\ndf_full (tutte le sequenze): {len(df_full)} sequenze")
print(f"df_no_unknown (solo paper): {len(df_no_unknown)} sequenze")
print(f"\nDifferenza: {len(df_full) - len(df_no_unknown)} sequenze rimosse")
print(f"Percentuale drop: {100 * (len(df_full) - len(df_no_unknown)) / len(df_full):.2f}%")


In [ ]:
# Breakdown per exercise
print("\n" + "-"*50)
print("Breakdown per Exercise:")

for ex in ["Squats", "Lunges", "Planks"]:
    full_count = len(df_full[df_full["exercise_paper"] == ex])
    clean_count = len(df_no_unknown[df_no_unknown["exercise_paper"] == ex])
    dropped = full_count - clean_count
    print(f"  {ex}: full={full_count}, clean={clean_count}, dropped={dropped}")


In [ ]:
# Crea maschera booleana per gli indici originali (da usare con ec3d_sequences.pkl)
# Questa maschera può essere usata per filtrare le sequenze negli esperimenti

# Carica ec3d_sequences.pkl per ottenere corrispondenza
SEQ_PATH = DATA_DIR / "ec3d_sequences.pkl"
with open(SEQ_PATH, "rb") as f:
    ec3d_seq = pickle.load(f)

meta_list = ec3d_seq["meta"]
print(f"Numero sequenze in ec3d_sequences.pkl: {len(meta_list)}")


In [ ]:
# Crea maschera: True = nella tassonomia paper, False = unknown
def is_in_paper_taxonomy(meta_item):
    """Verifica se una sequenza è nella tassonomia del paper."""
    ex = meta_item["exercise"]
    instr_id = meta_item["instruction_id"]
    
    # Normalizza nome exercise
    ex_paper = EXERCISE_NORMALIZE.get(ex, ex)
    
    return (ex_paper, instr_id) in PAPER_TAXONOMY

# Crea maschera booleana
mask_no_unknown = np.array([is_in_paper_taxonomy(m) for m in meta_list])

print(f"Sequenze nella tassonomia paper: {mask_no_unknown.sum()}")
print(f"Sequenze UNKNOWN: {(~mask_no_unknown).sum()}")

# Verifica coerenza
assert mask_no_unknown.sum() == len(df_no_unknown), "Mismatch tra maschera e df_no_unknown!"
print("\n✅ Maschera coerente con df_no_unknown")


In [ ]:
# Lista indici delle sequenze unknown
unknown_indices = np.where(~mask_no_unknown)[0]
print(f"Indici sequenze UNKNOWN: {unknown_indices.tolist()}")

# Dettaglio delle sequenze unknown
print("\nDettaglio sequenze UNKNOWN:")
for idx in unknown_indices:
    m = meta_list[idx]
    print(f"  [{idx}] {m['exercise']} - {m['instruction_name']} (id={m['instruction_id']}) - {m['subject']} - trial {m['trial_id']}")


In [ ]:
# Salva la maschera e le info per uso futuro
filter_data = {
    "mask_no_unknown": mask_no_unknown,
    "unknown_indices": unknown_indices.tolist(),
    "paper_taxonomy": PAPER_TAXONOMY,
    "exercise_normalize": EXERCISE_NORMALIZE,
    "subject_name_to_idx": SUBJECT_NAME_TO_IDX,  # Mapping corretto verificato col paper
    "subject_idx_to_name": SUBJECT_IDX_TO_NAME,
    "n_total": len(meta_list),
    "n_paper": int(mask_no_unknown.sum()),
    "n_unknown": int((~mask_no_unknown).sum()),
}

FILTER_PATH = DATA_DIR / "paper_taxonomy_filter.pkl"
with open(FILTER_PATH, "wb") as f:
    pickle.dump(filter_data, f)

print(f"\n✅ Salvato filtro: {FILTER_PATH}")
print(f"   - mask_no_unknown: array booleano di {len(mask_no_unknown)} elementi")
print(f"   - {filter_data['n_paper']} sequenze paper, {filter_data['n_unknown']} unknown")
print(f"   - subject_name_to_idx: {SUBJECT_NAME_TO_IDX}")


## H) Assert finali e verifica


In [ ]:
print("="*80)
print("ASSERT FINALI")
print("="*80)

# Totali per action
squats_total = len(df_no_unknown[df_no_unknown["exercise_paper"] == "Squats"])
lunges_total = len(df_no_unknown[df_no_unknown["exercise_paper"] == "Lunges"])
planks_total = len(df_no_unknown[df_no_unknown["exercise_paper"] == "Planks"])
overall_total = len(df_no_unknown)

print(f"\nSquats: {squats_total} (atteso: 132)")
print(f"Lunges: {lunges_total} (atteso: 127)")
print(f"Planks: {planks_total} (atteso: 103)")
print(f"TOTALE: {overall_total} (atteso: 362)")

# Assert
try:
    assert squats_total == 132, f"Squats mismatch: {squats_total} != 132"
    assert lunges_total == 127, f"Lunges mismatch: {lunges_total} != 127"
    assert planks_total == 103, f"Planks mismatch: {planks_total} != 103"
    assert overall_total == 362, f"Overall mismatch: {overall_total} != 362"
    print("\n✅ TUTTI GLI ASSERT PASSATI! I dati corrispondono al paper.")
except AssertionError as e:
    print(f"\n❌ ASSERT FALLITO: {e}")
    print("\nAnalisi differenze...")


In [ ]:
# Riepilogo finale
print("\n" + "="*80)
print("RIEPILOGO FINALE")
print("="*80)

print(f"""
📊 DATASET EC3D:
   - Totale sequenze: {len(df_full)}
   - Sequenze nel paper: {len(df_no_unknown)}
   - Sequenze UNKNOWN: {len(df_full) - len(df_no_unknown)}

📋 UNKNOWN IDENTIFICATI:
   - Combinazione: (SQUAT, instruction_id=10) → "Unknown" nel codice
   - Non presente nella tabella del paper
   - Totale: {len(seq_unknown)} sequenze

🎯 CONFRONTO CON PAPER:
   - Squats: {squats_total}/132 ({'✅ OK' if squats_total == 132 else '❌ MISMATCH'})
   - Lunges: {lunges_total}/127 ({'✅ OK' if lunges_total == 127 else '❌ MISMATCH'})
   - Planks: {planks_total}/103 ({'✅ OK' if planks_total == 103 else '❌ MISMATCH'})
   - TOTALE: {overall_total}/362 ({'✅ OK' if overall_total == 362 else '❌ MISMATCH'})

📁 FILE SALVATI:
   - {FILTER_PATH}
     → mask_no_unknown: array booleano per filtrare sequenze
     → unknown_indices: lista indici delle sequenze unknown
""")


In [ ]:
# Esempio di come usare il filtro negli esperimenti
print("\n" + "="*80)
print("ESEMPIO USO FILTRO NEGLI ESPERIMENTI")
print("="*80)

print("""
# Carica il filtro
with open('data/EC3D/paper_taxonomy_filter.pkl', 'rb') as f:
    filter_data = pickle.load(f)

mask = filter_data['mask_no_unknown']  # array booleano

# Carica le sequenze
with open('data/EC3D/ec3d_sequences.pkl', 'rb') as f:
    ec3d = pickle.load(f)

# Filtra per ottenere solo le sequenze del paper
sequences_clean = [ec3d['sequences'][i] for i in range(len(mask)) if mask[i]]
labels_clean = ec3d['labels'][mask]
meta_clean = [ec3d['meta'][i] for i in range(len(mask)) if mask[i]]

print(f"Sequenze originali: {len(ec3d['sequences'])}")
print(f"Sequenze filtrate (paper only): {len(sequences_clean)}")
""")


In [ ]:
# Verifica corrispondenza tra split_cross_subject.json e df
print("\n" + "="*80)
print("VERIFICA SPLIT TRAIN/TEST")
print("="*80)

# Conta sequenze per soggetto nel df_no_unknown
train_subj = ["Hugues", "Sena", "Isinsu"]
test_subj = ["Vidit"]

train_count = len(df_no_unknown[df_no_unknown["subject"].isin(train_subj)])
test_count = len(df_no_unknown[df_no_unknown["subject"].isin(test_subj)])

print(f"\nSequenze TRAIN (Subject 1,2,3): {train_count}")
print(f"Sequenze TEST (Subject 4): {test_count}")
print(f"Totale: {train_count + test_count}")

# Breakdown per subject
print("\nBreakdown per Subject (solo paper taxonomy):")
for subj_idx in [1, 2, 3, 4]:
    subj_name = SUBJECT_IDX_TO_NAME[subj_idx]
    count = len(df_no_unknown[df_no_unknown["subject_idx"] == subj_idx])
    role = "train" if subj_name in train_subj else "test"
    print(f"  Subject {subj_idx} ({subj_name}, {role}): {count} sequenze")
